In [1]:
from dataclasses import field, make_dataclass, dataclass
from typing import Any, Generic, Protocol, Self, TypeVar, runtime_checkable
from collections.abc import Callable, Mapping, MutableMapping, Sequence
from collections import namedtuple
import xml.etree.ElementTree as et
from kanji_time.utilities.class_property import classproperty

from functools import cached_property



In [2]:

Namespace = namedtuple('Namespace', ['tag', 'uri'])


T = TypeVar('T')
Factory = Callable[..., T]


In [3]:
class BetterQName(et.QName):
    """Isolate the namespace in a qualified name of the form "{uri}tag"."""

    def __init__(self, text_or_uri: str, tag:str = None):
        self._namespace: slice = slice(0, 0)
        self._tag: slice = slice(0, len(tag or text_or_uri))
        if tag:
            self._namespace = slice(1, len(text_or_uri) + 1)
            self._tag = slice(len(text_or_uri) + 2, None)
        elif text_or_uri[0] == '{':
            rb_index = text_or_uri.rfind('}', 1)
            if rb_index > 0:
                self._namespace = slice(1, rb_index)
                self._tag = slice(rb_index + 1, None)
        super().__init__(text_or_uri, tag)

    @property
    def namespace(self):
        return self.text[self._namespace]

    @property
    def tag(self):
        return self.text[self._tag]

assert BetterQName("foo").namespace == ""
assert BetterQName("foo").tag == "foo"
assert BetterQName("{foo").namespace == ""
assert BetterQName("{foo").tag == "{foo"
assert BetterQName("{}foo").namespace == ""
assert BetterQName("{}foo").tag == "foo"
assert BetterQName("{foo}").namespace == "foo"
assert BetterQName("{foo}").tag == ""
assert BetterQName("foo", "bar").namespace == "foo"
assert BetterQName("foo", "bar").tag == "bar"
assert BetterQName(BetterQName("foo", "bar").namespace, BetterQName("foo", "bar").tag) == BetterQName("foo", "bar")


In [20]:
T = TypeVar('T', covariant=True)
Factory = Callable[..., T]

@runtime_checkable
class TagFactory(Generic[T], Protocol):
    """
    Model creating a structure of some kind that contains XML tag data modeled as the type T.

    Methods
    -------

        * from_element - Create a tag instance from data in an ElementTree.Element instance.
        * child_from_element - Create a child tag instance to this factory's tak from an ElementTree.Element instance.
        * __call__ - Directly call the constructor function for type T

    .. only:: dev_notes
        
        * ElementTree is legacy to be removed in a later refinement - it's here to bootstrap the system.
        * It's highly likely that I'll implement ElementTree interfaces directly
        * child_from_element may choose to ignore or error out on an unexpected child tag.
          Crucially, it's up to the implementor to have a way to find a factory for that child tag

    """

    def from_element(self, element: et.Element) -> T:
        ...

    def child_from_element(self, element: et.Element) -> T | None:
        ...

    def __call__(self, *args, **kwargs) -> T:
        ...



In [21]:

@runtime_checkable
class TagInstance(Protocol):
    """
    Model data read from a tag in an XML model.

    This protocol is defined strictly as a convenient base for dataclasses modelling tag attributes.

    .. only:: dev_notes 

        * Should the *_text, *_children methods be encapsulated into a single type for text & children?
          well... maybe... I've resolved this by a single "append" for each and a read-only sequence accessor.

        * There should probably be callback hooks on here for "on_child" to intercept data into a custom structure.

        * It's up to the implementor to filter the sourcing ElementTree.Element instances for things that it's looking for.

    """

    @classproperty
    def factory(cls) -> TagFactory[Self]:
        ...

    @property
    def text(self) -> Sequence[str]:
        ...

    def append_text(self, text: str):
        ...

    @property
    def children(self) -> Sequence[Self]:
        ...

    def append_child(self, child: Self):
        ...



In [ ]:
U = TypeVar('U', covariant=True, bound=TagInstance)

class TagFactory_(TagFactory[U]):
    """
    Make tag instances using metadata provided at run time.
    
    T is the type holding the tag instance data.
    """

    @property
    def metadata(self) -> TagMetadata:
        _metadata =  SYMBOLS[self.tag_name].metadata
        assert id(_metadata.tag_factory) == id(self)
        return _metadata
    
    @property
    def namespace(self) -> str:
        return SYMBOLS[self.metadata.tag_name].namespace

    def __init__(self, metadata: TagMetadata, tag_base: type[U]):
        """
        Initialize a tag instance factory from its corresponding DTD metadata.

        The tag instance itself is a dataclass derived from the passed tag_base
        """
        self.tag_name = metadata.tag_name
        # A tag instance is a dataclass with string fields for the attributes derived from a custom base.
        # The TagInstance base is defined later - I'm aliasing it with a typevar T to avoid circular dependency issues.
        fields = [
            (attribute, str, field(default=''))
            for attribute in metadata.attribute_names
            ]
        self.tag_class = make_dataclass(
            metadata.tag_name, 
            fields, 
            bases=(tag_base,)
            )

    def from_element(self, element: et.Element) -> U:
        """
        Create a tag instance from data in an ElementTree.Element instance.

        It's redundant work, I'm bootstrapping from ElementTree first.
        """
        assert BetterQName(element.tag).tag == self.metadata.tag_name  # TODO: a real exception for this case
        attributes = {
            qname.tag: value
            for (qname, value) in [(BetterQName(k), v) for (k, v) in element.items()]
            if qname.namespace == self.namespace
            }
        return self.tag_class(**attributes)

    def child_from_element(self, element: et.Element) -> U | None:
        """
        Create a tag instance from data in an ElementTree.Element instance.

        It's redundant work, I'm bootstrapping from ElementTree first.
        """
        qtag = BetterQName(element.tag)
        if qtag.namespace != self.namespace:
            return None  # child isn't even in my namespace
        child_factory = SYMBOLS[qtag.tag].metadata.tag_factory
        assert child_factory is not None
        return child_factory.from_element(element)

    def __call__(self, *args, **kwargs) -> U:
        """Create a tag instance using its constructor function or by casting from ElementTree.Element."""
        return self.tag_class(*args, **kwargs)

assert issubclass(TagFactory_, TagFactory)



In [28]:

class TagInstance_:
    """Provide a common root class for different dataclass representations of XML tags."""

    def __init_subclass__(cls, /, factory: TagFactory[Self], **kwargs):    
        print(f"{cls.__name__}.__init_subclass__")
        super().__init_subclass__(**kwargs)
        cls._factory: TagFactory[Self] = factory

    @classproperty
    def factory(cls) -> TagFactory[Self]:
        return cls._factory

    @property
    def text(self) -> Sequence[str]:
        raise NotImplementedError(f"{self.__class__.__name__}.text")

    def append_text(self, text: str):
        raise NotImplementedError(f"{self.__class__.__name__}.append_text")

    @property
    def children(self) -> Sequence[Self]:
        raise NotImplementedError(f"{self.__class__.__name__}.children")

    def append_child(self, child: Self):
        raise NotImplementedError(f"{self.__class__.__name__}.append_child")

    @classmethod
    def from_element(cls, element: et.Element):
        return cls.factory.from_element(element)
    
    @classmethod
    def child_from_element(cls, child_element: et.Element):
        return cls.factory.child_from_element(child_element)


In [ ]:


@runtime_checkable
class TagMetadata(Generic[T], Protocol):
    """
    Model the content of an XML DTD for a particular tag independent of namespaces.

    This model is incomplete and super simple - yet exactly enough to do what I need.

    The TagMetadata describes its instance data as properties to allow implementors freedom to 
    represent them as they see fit.

        *tag_name* - the unqualified name of the XML tag as given in <!ELEMENT *tag_name* ... >
        
        *attributes* - the unqualified names of the XML tag's attributes as given in <!ATTLIST *tag_name* *attr_name* *attr_type* *attr_dflt* *attr_name* *attr_type* *attr_dflt* ... >

        *children* - the unqualified name of a the XML tag's child entities as given in <!ELEMENT *tag_name* (*child_name* ...)>
    """
    @property
    def tag_name(self) -> str:
        """Expose the unqualified name of the described XML tag."""
        ...

    @property
    def attribute_names(self) -> Sequence[str]:
        """Expose the unqualified attribute names for the described XML tag."""
        ...

    @property
    def child_tag_names(self) -> Sequence[str]:
        """Expose the unqualified child tag names for the described XML tag."""
        ...

    @property
    def tag_factory(self) -> TagFactory[T]:
        """Provide a construction function for classes containing instances of the type described by this metadata."""
        ...

    @property
    def continuation(self) -> Self | None:
        """Link source metadata when this metadata is a refinement of other metadata."""
        ...



In [ ]:
@dataclass 
class TagSymbol:
    namespace: str
    name: str
    metadata: TagMetadata

FrozenSymbolTable = dict[str, TagSymbol]

SYMBOLS: FrozenSymbolTable = {}
